# 🚀 Lunar Lander Reinforcement Learning & Reward Hacking

This interactive notebook demonstrates training an AI rocket lander using Proximal Policy Optimization (PPO) in Gymnasium's `LunarLander-v3`.

We explore **reward shaping** and show how an innocent-looking survival bonus (`+1/frame`) creates a classic **reward hacking (specification gaming)** loophole where the agent learns to hover indefinitely instead of landing.

### 1. Install Dependencies

In [ ]:
!pip install -q "gymnasium[box2d]>=0.29.0" stable-baselines3 torch moviepy rich

### 2. Custom Reward Wrapper
Here is the wrapper that allows switching between `default`, `hack`, and `fixed` modes.

In [ ]:
import math
import gymnasium as gym
import numpy as np

class CustomRewardWrapper(gym.Wrapper):
    def __init__(self, env, reward_mode='default', step_penalty=0.05):
        super().__init__(env)
        self.reward_mode = reward_mode
        self.step_penalty = step_penalty
        self.prev_shaping = None

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.prev_shaping = self._calculate_shaping(obs)
        return obs, info

    def _calculate_shaping(self, obs):
        x, y, vx, vy, angle, _, leg_l, leg_r = obs[:8]
        dist = math.sqrt(x*x + y*y)
        speed = math.sqrt(vx*vx + vy*vy)
        tilt = abs(angle)
        return -100.0 * dist - 100.0 * speed - 100.0 * tilt + 10.0 * leg_l + 10.0 * leg_r

    def step(self, action):
        obs, reward, terminated, truncated, info = self.env.step(action)
        shaped_reward = reward

        if self.reward_mode == 'hack':
            # +1 per frame causes hover gaming
            shaped_reward += 1.0
        elif self.reward_mode == 'fixed':
            current = self._calculate_shaping(obs)
            delta = current - self.prev_shaping if self.prev_shaping is not None else 0.0
            self.prev_shaping = current
            shaped_reward = reward if terminated else (delta - self.step_penalty)

        return obs, shaped_reward, terminated, truncated, info

### 3. Baseline: Untrained Random Agent

In [ ]:
env = gym.make('LunarLander-v3')
obs, info = env.reset(seed=42)
total_reward = 0
done = False

while not done:
    action = env.action_space.sample()
    obs, reward, terminated, truncated, info = env.step(action)
    total_reward += reward
    done = terminated or truncated

print(f"Untrained Random Agent Score: {total_reward:.2f} (Crashed)")
env.close()

### 4. Train PPO Agent (Default vs Hack vs Fixed)

In [ ]:
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor

# Train default solver (~150,000 steps)
env = Monitor(gym.make('LunarLander-v3'))
model_default = PPO('MlpPolicy', env, verbose=0, seed=42)
print("Training Default PPO Agent...")
model_default.learn(total_timesteps=150000)
print("Training Complete!")